# Module 01: GRPO — Group Relative Policy Optimization from Scratch

## Learning Objectives

By completing this notebook, you will:
1. Implement the full GRPO update step using only numpy, with no black-box library calls
2. Explain why GRPO computes advantages relative to a *group* of sampled completions rather than a learned value baseline
3. Demonstrate the effect of the clipped surrogate objective and the KL divergence penalty on training stability

## Prerequisites
- Familiarity with probability distributions and expectation
- Basic understanding of what a policy is: a distribution over actions
- Python and numpy comfort

## Estimated Time: 15 minutes

---

## Why GRPO Matters

Standard PPO (Proximal Policy Optimization) needs a *critic network* — a separate model trained to predict how good a state is. For LLMs, that means training a second model as large as the policy model itself. GRPO eliminates the critic entirely.

Instead of asking "how good is this state?", GRPO asks: **"how much better was this completion than the other completions I sampled at the same time?"** The group of sampled completions *is* the baseline.

This is the core idea that makes GRPO practical for training large language models and reasoning agents.

## Setup and Imports

All dependencies are standard scientific Python. No deep learning framework is required — we build every component ourselves.

In [ ]:
# Purpose: Import all dependencies for this notebook
# Key Concept: This notebook uses only numpy and matplotlib — no deep learning framework needed

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Set random seed for reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'reinforce': '#2ecc71',   # green — reward goes up
    'suppress':  '#e74c3c',   # red — reward goes down
    'neutral':   '#95a5a6',   # grey
    'policy':    '#3498db',   # blue — policy values
    'reference': '#e67e22',   # orange — reference policy
}

print("Dependencies loaded. numpy", np.__version__)

---

## Section 1 — The Policy as a Probability Distribution

In GRPO (and in LLM training generally), a **policy** is a probability distribution over possible responses to a prompt. For a language model, this distribution has billions of parameters. Here, we work with a stripped-down version: a discrete distribution over a small menu of candidate responses.

We represent the policy as a weight vector. The softmax function converts raw weights (logits) into valid probabilities that sum to 1:

$$\pi_\theta(a) = \frac{e^{\theta_a}}{\sum_j e^{\theta_j}}$$

Sampling from this distribution gives us our group of completions. The cell below defines `softmax` and `sample_completions`, then demos both on a set of four SQL query patterns.

In [ ]:
# Purpose: Define the policy representation and sampling machinery
# Key Concept: A policy is a categorical distribution. Raw weights -> softmax -> probabilities.

def softmax(logits):
    """
    Convert logits to a valid probability distribution.

    Subtracts the max before exponentiation for numerical stability.
    Without this, large logits cause overflow to inf.

    Parameters
    ----------
    logits : np.ndarray, shape (n_actions,)
        Raw unnormalized scores

    Returns
    -------
    np.ndarray, shape (n_actions,)
        Probabilities that sum to 1
    """
    # Numerical stability: subtract max so largest exponent is e^0 = 1
    shifted = logits - np.max(logits)
    exp_vals = np.exp(shifted)
    return exp_vals / exp_vals.sum()


def sample_completions(logits, n_completions, rng):
    """
    Sample a group of completion indices from the current policy.

    Parameters
    ----------
    logits : np.ndarray, shape (n_actions,)
        Current policy logits
    n_completions : int
        Number of completions to sample (G in the GRPO paper)
    rng : np.random.Generator

    Returns
    -------
    indices : np.ndarray, shape (n_completions,)
        Sampled action indices
    probs : np.ndarray, shape (n_actions,)
        The policy probabilities used for sampling
    """
    probs = softmax(logits)
    indices = rng.choice(len(logits), size=n_completions, p=probs)
    return indices, probs


# Four candidate SQL patterns a model might produce
RESPONSES = [
    "SELECT * FROM orders WHERE status = 'open'",
    "SELECT id, amount FROM orders WHERE status = 'open' ORDER BY amount DESC",
    "SELECT id, amount FROM orders WHERE status = 'open' AND amount > 100",
    "SELECT id FROM orders WHERE status = 'open' LIMIT 10",
]

# Start with uniform logits — no preference yet
initial_logits = np.zeros(len(RESPONSES))
probs = softmax(initial_logits)

print("Initial policy (uniform):")
for i, (resp, p) in enumerate(zip(RESPONSES, probs)):
    print(f"  [{i}] p={p:.3f}  {resp[:55]}")

---

## Section 2 — Advantage Estimation: The Heart of GRPO

Once we have a group of $G$ sampled completions with rewards $r_1, \ldots, r_G$, we compute the **advantage** for each completion:

$$A_i = \frac{r_i - \mu_r}{\sigma_r}$$

where $\mu_r$ is the mean reward across the group and $\sigma_r$ is the standard deviation.

This is just **z-score normalization** of the rewards. The result:
- Completions with above-average reward get **positive advantage** — we want more of these
- Completions with below-average reward get **negative advantage** — we want less of these
- No external value model needed: the group *is* the baseline

A small epsilon is added to the denominator to prevent division by zero when all rewards are identical.

The next cell implements `compute_advantages` and walks through the arithmetic step by step for the example rewards `[0.3, 0.5, 0.7, 0.9]`.

In [ ]:
# Purpose: Implement GRPO advantage computation and show the step-by-step arithmetic
# Key Concept: Advantage = z-score of reward. The group is its own baseline.

def compute_advantages(rewards):
    """
    Compute GRPO advantages as z-scores of rewards within the group.

    Parameters
    ----------
    rewards : np.ndarray, shape (G,)
        Scalar reward for each sampled completion

    Returns
    -------
    advantages : np.ndarray, shape (G,)
        Normalized advantages. Mean ~0, std ~1.
    """
    mu = rewards.mean()
    sigma = rewards.std()

    # epsilon prevents division by zero when all rewards are equal
    eps = 1e-8
    advantages = (rewards - mu) / (sigma + eps)
    return advantages


# --- Worked Example ---
# Four completions scored by a reward function (higher = better SQL quality)
rewards_example = np.array([0.3, 0.5, 0.7, 0.9])
advantages_example = compute_advantages(rewards_example)

mu    = rewards_example.mean()
sigma = rewards_example.std()

print("=== Advantage Computation: Step by Step ===")
print(f"\nGroup rewards:       {rewards_example}")
print(f"Mean reward (mu):    {mu:.4f}")
print(f"Std  reward (sigma): {sigma:.4f}")
print()
print(f"{'Completion':>12}  {'Reward':>8}  {'(r - mu)':>10}  {'Advantage':>10}  {'Effect'}")
print("-" * 62)
for i, (r, a) in enumerate(zip(rewards_example, advantages_example)):
    effect = "REINFORCE" if a > 0 else "SUPPRESS"
    print(f"  [{i}]         {r:>8.3f}  {(r - mu):>10.4f}  {a:>10.4f}  {effect}")

The table makes the key point concrete: completions 2 and 3 (rewards 0.7 and 0.9) have positive advantage and will be reinforced; completions 0 and 1 (rewards 0.3 and 0.5) have negative advantage and will be suppressed.

The following cell renders this as a side-by-side bar chart: raw rewards on the left, normalized advantages on the right. Green bars go up in probability; red bars go down.

In [ ]:
# Purpose: Visualize which completions are reinforced vs suppressed
# Key Concept: Positive advantage = increase probability; negative = decrease

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bar_colors = [COLORS['suppress'] if a <= 0 else COLORS['reinforce']
              for a in advantages_example]

# Left: raw rewards with group mean line
ax = axes[0]
bars = ax.bar(range(4), rewards_example, color=bar_colors, edgecolor='white', linewidth=1.5)
ax.axhline(mu, color='black', linestyle='--', linewidth=1.5, label=f'Group mean = {mu:.2f}')
ax.set_xticks(range(4))
ax.set_xticklabels([f'Completion {i}' for i in range(4)])
ax.set_ylabel('Reward', fontsize=12)
ax.set_title('Raw Rewards per Completion', fontsize=13)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
for bar, r in zip(bars, rewards_example):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{r:.1f}', ha='center', fontsize=11, fontweight='bold')

# Right: normalized advantages
ax = axes[1]
bars = ax.bar(range(4), advantages_example, color=bar_colors, edgecolor='white', linewidth=1.5)
ax.axhline(0, color='black', linewidth=1.2)
ax.set_xticks(range(4))
ax.set_xticklabels([f'Completion {i}' for i in range(4)])
ax.set_ylabel('Advantage (z-score)', fontsize=12)
ax.set_title('Advantages: Who Gets Reinforced?', fontsize=13)
for bar, a in zip(bars, advantages_example):
    ypos = a + 0.05 if a >= 0 else a - 0.12
    ax.text(bar.get_x() + bar.get_width() / 2, ypos,
            f'{a:+.2f}', ha='center', fontsize=11, fontweight='bold')

reinforce_patch = mpatches.Patch(color=COLORS['reinforce'], label='Reinforce (A > 0)')
suppress_patch  = mpatches.Patch(color=COLORS['suppress'],  label='Suppress  (A < 0)')
axes[1].legend(handles=[reinforce_patch, suppress_patch], fontsize=11)

plt.suptitle('GRPO Step 1: Reward -> Advantage', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## Section 3 — The Clipped Surrogate Objective

Now that we have advantages, we need to update the policy. PPO and GRPO share the same clipped objective to prevent the update from being too large:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_i \left[ \min\left( \rho_i A_i,\ \text{clip}(\rho_i,\ 1-\varepsilon,\ 1+\varepsilon) \cdot A_i \right) \right]$$

where the **probability ratio** is:

$$\rho_i = \frac{\pi_\theta(a_i)}{\pi_{\text{old}}(a_i)}$$

The ratio $\rho_i$ measures how much the current policy has moved away from the policy that generated the completions. When $\rho_i = 1$, the policy has not changed at all.

**Why clip?** Without clipping, a single large gradient step could increase $\pi_\theta(a_i)$ by a factor of 10 or more. The clip limits the effective ratio to the range $[1-\varepsilon, 1+\varepsilon]$, keeping updates conservative. Typical $\varepsilon = 0.2$.

In this notebook we implement the update as a direct logit modification rather than gradient descent — it produces the same directional effect and makes the mechanics fully transparent.

In [ ]:
# Purpose: Implement the clipped surrogate objective
# Key Concept: clip(rho, 1-eps, 1+eps) prevents catastrophically large policy updates

def clipped_surrogate(ratio, advantage, epsilon=0.2):
    """
    Compute the PPO/GRPO clipped surrogate objective for a single completion.

    Parameters
    ----------
    ratio : float
        Probability ratio pi_theta(a) / pi_old(a)
    advantage : float
        Normalized advantage for this completion
    epsilon : float
        Clip range. Ratio is constrained to [1 - epsilon, 1 + epsilon].

    Returns
    -------
    float
        Objective value. Maximize this.
    """
    # Unclipped term: ratio * advantage
    unclipped = ratio * advantage

    # Clipped term: restrict ratio movement, then multiply by same advantage
    clipped_ratio = np.clip(ratio, 1.0 - epsilon, 1.0 + epsilon)
    clipped = clipped_ratio * advantage

    # Take the minimum: pessimistic bound prevents over-optimistic updates
    return min(unclipped, clipped)

With `clipped_surrogate` defined, we can now build `grpo_update` — the full policy update step. It loops over the sampled completions, computes the clipped objective for each, accumulates a per-action update signal, and adjusts logits proportionally.

In [ ]:
# Purpose: Implement the full GRPO policy update step
# Key Concept: Update logits in proportion to the clipped surrogate objective per action

def grpo_update(logits, sampled_indices, old_probs, advantages,
                epsilon=0.2, learning_rate=0.5):
    """
    Apply one GRPO update step to the policy logits.

    Instead of automatic differentiation, we compute the update direction
    analytically: increase logits for high-advantage actions, decrease for low.
    The magnitude is controlled by the clipped surrogate.

    Parameters
    ----------
    logits : np.ndarray, shape (n_actions,)
        Current policy logits
    sampled_indices : np.ndarray, shape (G,)
        Which action was sampled for each completion
    old_probs : np.ndarray, shape (n_actions,)
        Policy probabilities at sampling time (pi_old)
    advantages : np.ndarray, shape (G,)
        Per-completion advantages
    epsilon : float
        Clip range for probability ratio
    learning_rate : float
        Step size for logit update

    Returns
    -------
    new_logits : np.ndarray, shape (n_actions,)
        Updated logits
    objectives : list of float
        Clipped surrogate objective value per completion
    """
    new_logits    = logits.copy()
    current_probs = softmax(logits)

    # Accumulate per-action update signals across all sampled completions
    action_update = np.zeros(len(logits))
    action_count  = np.zeros(len(logits))
    objectives    = []

    for idx, adv in zip(sampled_indices, advantages):
        # Probability ratio: how much has this action's probability changed?
        ratio = current_probs[idx] / (old_probs[idx] + 1e-10)

        # Clipped surrogate objective for this completion
        obj = clipped_surrogate(ratio, adv, epsilon)
        objectives.append(obj)

        # Accumulate: positive objective = increase logit, negative = decrease
        action_update[idx] += obj
        action_count[idx]  += 1

    # Average updates when the same action was sampled more than once
    mask = action_count > 0
    averaged_update = np.where(mask, action_update / np.maximum(action_count, 1), 0.0)

    new_logits += learning_rate * averaged_update

    return new_logits, objectives


# --- Verify: run one update on the worked example ---
logits_0    = np.zeros(4)
indices_0   = np.array([0, 1, 2, 3])  # one sample of each action
old_probs_0 = softmax(logits_0)

new_logits_0, objs_0 = grpo_update(
    logits_0, indices_0, old_probs_0, advantages_example
)

print("=== One GRPO Update Step ===")
print(f"\n{'Action':>8}  {'Old prob':>10}  {'Advantage':>10}  {'Objective':>10}  {'New logit':>10}  {'New prob':>10}")
print("-" * 70)
new_probs_0 = softmax(new_logits_0)
for i in range(4):
    print(f"  [{i}]      {old_probs_0[i]:>10.4f}  {advantages_example[i]:>10.4f}  "
          f"{objs_0[i]:>10.4f}  {new_logits_0[i]:>10.4f}  {new_probs_0[i]:>10.4f}")

---

## Section 4 — Training Simulation: GRPO Over Multiple Steps

Now we run the full training loop. At each step:

1. Sample G=4 completions from the current policy
2. Score each completion with the reward function
3. Compute advantages (z-score normalization within the group)
4. Apply the GRPO update
5. Log the mean reward and policy entropy

**Policy entropy** measures how spread-out the policy is:
$$H(\pi) = -\sum_a \pi(a) \log \pi(a)$$

High entropy = uniform distribution = no preference yet. Low entropy = the model has committed to a small number of actions. Entropy should decrease as training progresses — the model learns which SQL pattern works best.

The next cell defines the reward function and the entropy utility.

In [ ]:
# Purpose: Define the reward function for the SQL task and the entropy utility
# Key Concept: The reward function is the only task-specific component in GRPO

# Ground-truth reward for each SQL pattern (higher = better query quality)
# In real LLM training this comes from a verifier or preference model
TRUE_REWARDS = np.array([0.2, 0.9, 0.6, 0.3])

def reward_function(action_index):
    """
    Return the reward for selecting a given SQL response pattern.

    In production GRPO, this is a learned reward model or a rule-based verifier.
    Here we use a fixed lookup table so the correct answer is knowable.

    Parameters
    ----------
    action_index : int
        Index into RESPONSES list

    Returns
    -------
    float
        Scalar reward in [0, 1]
    """
    # Small noise simulates reward model variance
    noise = rng.normal(0, 0.02)
    return float(np.clip(TRUE_REWARDS[action_index] + noise, 0.0, 1.0))


def policy_entropy(logits):
    """
    Compute Shannon entropy of the policy distribution.

    Parameters
    ----------
    logits : np.ndarray
        Policy logits

    Returns
    -------
    float
        Entropy in nats. Maximum = log(n_actions) for uniform policy.
    """
    probs = softmax(logits)
    return -np.sum(probs * np.log(np.clip(probs, 1e-10, 1.0)))


print("Reward lookup table:")
for i, resp in enumerate(RESPONSES):
    print(f"  [{i}] reward={TRUE_REWARDS[i]:.2f}  {resp[:55]}")
print(f"\nMaximum entropy (uniform policy): {np.log(4):.4f} nats")

Now we run 60 GRPO steps and collect reward and entropy histories at each step.

In [ ]:
# Purpose: Run the full GRPO training loop and collect per-step metrics
# Key Concept: Each iteration is one GRPO step — sample -> score -> advantage -> update

N_STEPS       = 60    # number of GRPO update steps
G             = 4     # group size: completions sampled per step
EPSILON       = 0.2   # clip range
LEARNING_RATE = 0.4   # logit update step size

logits = np.zeros(4)  # start from a uniform policy

history = {'mean_reward': [], 'entropy': [], 'probs': []}

for step in range(N_STEPS):
    # Step 1: Sample G completions from current policy
    sampled_indices, old_probs = sample_completions(logits, G, rng)

    # Step 2: Score each completion
    rewards = np.array([reward_function(idx) for idx in sampled_indices])

    # Step 3: Compute advantages (z-score normalization within the group)
    advantages = compute_advantages(rewards)

    # Step 4: Apply GRPO update
    logits, _ = grpo_update(
        logits, sampled_indices, old_probs, advantages,
        epsilon=EPSILON, learning_rate=LEARNING_RATE
    )

    # Step 5: Log metrics
    history['mean_reward'].append(rewards.mean())
    history['entropy'].append(policy_entropy(logits))
    history['probs'].append(softmax(logits).copy())

print("Training complete. Final policy:")
final_probs = softmax(logits)
for i, (p, resp) in enumerate(zip(final_probs, RESPONSES)):
    marker = "<-- selected" if p == final_probs.max() else ""
    print(f"  [{i}] p={p:.4f}  reward={TRUE_REWARDS[i]:.2f}  {marker}")

Three charts tell the story of training: reward climbing toward the best possible value, entropy falling as the policy commits, and per-action probabilities tracking which SQL pattern the policy has learned to prefer.

In [ ]:
# Purpose: Visualize training dynamics — reward curve, entropy decay, and probability trajectories
# Key Concept: Rising reward + falling entropy = the policy is learning

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

rewards_arr = np.array(history['mean_reward'])
window      = 5
smoothed    = np.convolve(rewards_arr, np.ones(window) / window, mode='valid')

# --- Reward over time ---
ax = axes[0]
ax.plot(rewards_arr, color=COLORS['neutral'], alpha=0.4, linewidth=1, label='Per-step mean')
ax.plot(range(window - 1, N_STEPS), smoothed,
        color=COLORS['reinforce'], linewidth=2.5, label=f'{window}-step smooth')
ax.axhline(TRUE_REWARDS.max(), color='black', linestyle='--',
           linewidth=1.2, label=f'Best reward = {TRUE_REWARDS.max()}')
ax.set_xlabel('GRPO Step', fontsize=12)
ax.set_ylabel('Mean Group Reward', fontsize=12)
ax.set_title('Reward Improves Over Training', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)

# --- Entropy over time ---
ax = axes[1]
entropy_arr = np.array(history['entropy'])
ax.plot(entropy_arr, color=COLORS['policy'], linewidth=2.5)
ax.axhline(np.log(4), color='black', linestyle='--', linewidth=1.2,
           label=f'Max entropy = {np.log(4):.2f}')
ax.axhline(0, color='grey', linestyle=':', linewidth=1.0, label='Min entropy = 0')
ax.set_xlabel('GRPO Step', fontsize=12)
ax.set_ylabel('Policy Entropy (nats)', fontsize=12)
ax.set_title('Entropy Decreases as Policy Commits', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(-0.1, np.log(4) + 0.2)

# --- Probability evolution ---
ax = axes[2]
probs_arr     = np.array(history['probs'])   # shape: (N_STEPS, 4)
action_colors = [COLORS['suppress'], COLORS['reinforce'], COLORS['neutral'], COLORS['reference']]
for i in range(4):
    ax.plot(probs_arr[:, i], color=action_colors[i], linewidth=2,
            label=f'Action {i} (r={TRUE_REWARDS[i]:.1f})')
ax.set_xlabel('GRPO Step', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Policy Probability per Action', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)

plt.suptitle('GRPO Training Dynamics — SQL Pattern Selection Task',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Section 5 — KL Divergence Penalty: Keeping the Policy Grounded

Without any regularization, the policy can collapse to a single action even when that action's superiority is uncertain. This is **mode collapse**: the model becomes maximally confident after just a few steps.

GRPO adds a KL divergence penalty to prevent this:

$$L^{\text{GRPO}}(\theta) = L^{\text{CLIP}}(\theta) - \beta \cdot D_{\text{KL}}\left[\pi_\theta \,\|\, \pi_{\text{ref}}\right]$$

The KL term measures how far the updated policy $\pi_\theta$ has drifted from a fixed reference policy $\pi_{\text{ref}}$ (typically the initial pretrained model):

$$D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}}) = \sum_a \pi_\theta(a) \log \frac{\pi_\theta(a)}{\pi_{\text{ref}}(a)}$$

The coefficient $\beta$ controls the tradeoff:
- **Large $\beta$**: Policy stays close to the reference — safe but slow to learn
- **Small $\beta$**: Policy is free to explore — learns fast but risks mode collapse
- **$\beta = 0$**: No penalty — pure reward maximization, prone to collapse

The next cell implements `kl_divergence` and the penalized update `grpo_update_with_kl`.

In [ ]:
# Purpose: Implement KL divergence and the KL-penalized GRPO update
# Key Concept: KL penalty adds a restoring force that pulls the policy back toward the reference

def kl_divergence(p, q):
    """
    Compute KL divergence KL(p || q) = sum_a p(a) * log(p(a) / q(a)).

    KL is always >= 0. KL = 0 only when p == q exactly.
    It is NOT symmetric: KL(p||q) != KL(q||p) in general.

    Parameters
    ----------
    p : np.ndarray
        Distribution we measure divergence FROM (current policy)
    q : np.ndarray
        Reference distribution

    Returns
    -------
    float
        KL divergence in nats
    """
    p_safe = np.clip(p, 1e-10, 1.0)
    q_safe = np.clip(q, 1e-10, 1.0)
    return float(np.sum(p_safe * np.log(p_safe / q_safe)))


def grpo_update_with_kl(logits, ref_logits, sampled_indices, old_probs, advantages,
                        epsilon=0.2, beta=0.1, learning_rate=0.5):
    """
    GRPO update step with KL divergence penalty.

    After the clipped surrogate update, the logits are pulled back toward the
    reference proportional to how far they have drifted, scaled by beta.

    Parameters
    ----------
    logits : np.ndarray
        Current policy logits
    ref_logits : np.ndarray
        Reference policy logits (frozen, does not change)
    sampled_indices : np.ndarray, shape (G,)
    old_probs : np.ndarray, shape (n_actions,)
    advantages : np.ndarray, shape (G,)
    epsilon : float
        Clip range
    beta : float
        KL penalty coefficient
    learning_rate : float

    Returns
    -------
    new_logits : np.ndarray
    kl : float
        KL divergence from reference after the update
    """
    # Apply clipped surrogate update first
    new_logits, _ = grpo_update(
        logits, sampled_indices, old_probs, advantages,
        epsilon=epsilon, learning_rate=learning_rate
    )

    # Measure drift from reference
    current_probs = softmax(new_logits)
    ref_probs     = softmax(ref_logits)
    kl = kl_divergence(current_probs, ref_probs)

    # Pull logits back toward reference proportional to drift magnitude
    # Approximates subtracting beta * gradient(KL) from the update
    kl_correction = beta * (new_logits - ref_logits)
    new_logits    = new_logits - kl_correction

    return new_logits, kl


# Verify KL properties
p_uniform = softmax(np.zeros(4))
p_peaked  = softmax(np.array([0.0, 5.0, 0.0, 0.0]))

print("KL divergence properties:")
print(f"  KL(uniform || uniform) = {kl_divergence(p_uniform, p_uniform):.6f}  (should be 0)")
print(f"  KL(peaked  || uniform) = {kl_divergence(p_peaked,  p_uniform):.4f}  (peaked is far from uniform)")
print(f"  KL(uniform || peaked ) = {kl_divergence(p_uniform, p_peaked ):.4f}  (KL is not symmetric)")

Now we run the same training task with three different beta values — 0.0 (no penalty), 0.05 (light), and 0.3 (strong) — to show how the penalty affects reward, policy drift, and entropy.

In [ ]:
# Purpose: Compare training dynamics across three KL penalty strengths
# Key Concept: beta controls the tradeoff between reward optimization and staying near the reference

BETA_VALUES = [0.0, 0.05, 0.3]
N_STEPS_KL  = 80
ref_logits  = np.zeros(4)   # reference = uniform (the initial model)

results_kl = {}

for beta in BETA_VALUES:
    logits       = np.zeros(4)
    step_rewards = []
    step_kl      = []
    step_entropy = []

    for step in range(N_STEPS_KL):
        sampled_indices, old_probs = sample_completions(logits, G, rng)
        rewards    = np.array([reward_function(idx) for idx in sampled_indices])
        advantages = compute_advantages(rewards)

        logits, kl = grpo_update_with_kl(
            logits, ref_logits, sampled_indices, old_probs, advantages,
            epsilon=EPSILON, beta=beta, learning_rate=LEARNING_RATE
        )

        step_rewards.append(rewards.mean())
        step_kl.append(kl)
        step_entropy.append(policy_entropy(logits))

    results_kl[beta] = {
        'rewards':     np.array(step_rewards),
        'kl':          np.array(step_kl),
        'entropy':     np.array(step_entropy),
        'final_probs': softmax(logits),
    }

print("Final policy probabilities by beta value:")
for beta in BETA_VALUES:
    probs = results_kl[beta]['final_probs']
    ent   = results_kl[beta]['entropy'][-1]
    print(f"  beta={beta:.2f}: {np.round(probs, 3)}  (entropy={ent:.3f})")

The three charts below show: (1) how quickly reward improves, (2) how far each policy drifts from the reference, and (3) whether entropy collapses early — the signature of mode collapse at beta=0.

In [ ]:
# Purpose: Visualize the effect of different KL penalty strengths on training
# Key Concept: Without KL penalty (beta=0) entropy collapses rapidly — mode collapse

beta_colors = {0.0: COLORS['suppress'], 0.05: COLORS['reinforce'], 0.3: COLORS['policy']}
beta_labels = {
    0.0:  'beta=0.00 (no penalty)',
    0.05: 'beta=0.05 (light penalty)',
    0.3:  'beta=0.30 (strong penalty)',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for beta in BETA_VALUES:
    data       = results_kl[beta]
    color      = beta_colors[beta]
    label      = beta_labels[beta]
    w          = 5
    smoothed_r = np.convolve(data['rewards'], np.ones(w) / w, mode='valid')

    axes[0].plot(range(w - 1, N_STEPS_KL), smoothed_r, color=color, linewidth=2.2, label=label)
    axes[1].plot(data['kl'],      color=color, linewidth=2.2, label=label)
    axes[2].plot(data['entropy'], color=color, linewidth=2.2, label=label)

axes[0].axhline(TRUE_REWARDS.max(), color='black', linestyle='--', linewidth=1.2,
                label=f'Best reward = {TRUE_REWARDS.max()}')
axes[0].set_xlabel('GRPO Step', fontsize=12)
axes[0].set_ylabel('Mean Group Reward (smoothed)', fontsize=12)
axes[0].set_title('Reward vs. Beta', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.05)

axes[1].set_xlabel('GRPO Step', fontsize=12)
axes[1].set_ylabel('KL Divergence from Reference', fontsize=12)
axes[1].set_title('Policy Drift from Reference', fontsize=13)
axes[1].legend(fontsize=9)

axes[2].axhline(np.log(4), color='black', linestyle='--', linewidth=1.2,
                label='Max entropy (uniform)')
axes[2].set_xlabel('GRPO Step', fontsize=12)
axes[2].set_ylabel('Policy Entropy (nats)', fontsize=12)
axes[2].set_title('Entropy: Mode Collapse Visible at beta=0', fontsize=13)
axes[2].legend(fontsize=9)

plt.suptitle('Effect of KL Penalty Coefficient (beta) on GRPO Training',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Section 6 — Exercises

The following exercises require you to write code, not just run it. Each cell starts with placeholders set to `None` or `pass`. Running without modification will fail the auto-graded tests by design.

---

### Exercise 1.1: Implement Advantage Computation from Scratch

**Task:** Implement `my_compute_advantages` without calling `compute_advantages` or numpy's built-in `std`. Compute mean and standard deviation using only `np.sum`, `np.sqrt`, and arithmetic.

**Expected Output:** The result must match `compute_advantages` to within floating-point tolerance on both the standard test case and the edge case where all rewards are equal.

**Hints:**

<details>
<summary>Hint 1</summary>
Mean is sum divided by count: `mu = np.sum(rewards) / len(rewards)`
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Standard deviation is the square root of the mean squared deviation: `sigma = np.sqrt(np.sum((rewards - mu)**2) / len(rewards))`. Then divide by `sigma + 1e-8`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Implement advantage computation using only np.sum, np.sqrt, and arithmetic.
# Do NOT use np.mean, np.std, or compute_advantages.

def my_compute_advantages(rewards):
    """
    Compute GRPO advantages as z-scores of the reward array.

    Parameters
    ----------
    rewards : np.ndarray, shape (G,)

    Returns
    -------
    np.ndarray, shape (G,)
    """
    # Step 1: Compute mean reward using only np.sum
    mu = None  # replace with your implementation

    # Step 2: Compute standard deviation using only np.sum and np.sqrt
    sigma = None  # replace with your implementation

    # Step 3: Normalize (z-score) with epsilon for numerical safety
    eps = 1e-8
    advantages = None  # replace with your implementation

    return advantages

Run the cell below to check your implementation. It tests both a normal reward array and the edge case where all rewards are identical.

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_1_1():
    test_rewards = np.array([0.3, 0.5, 0.7, 0.9])

    result = my_compute_advantages(test_rewards)

    assert result is not None, (
        "my_compute_advantages returned None. "
        "Replace the 'None' placeholders with real computations."
    )
    assert isinstance(result, np.ndarray), (
        f"Expected np.ndarray, got {type(result)}. "
        "Make sure your advantages variable is a numpy array."
    )
    assert result.shape == (4,), (
        f"Expected shape (4,), got {result.shape}. "
        "Advantages must have the same length as rewards."
    )

    expected = compute_advantages(test_rewards)
    assert np.allclose(result, expected, atol=1e-6), (
        f"Values do not match reference implementation.\n"
        f"  Your output:     {np.round(result, 4)}\n"
        f"  Expected output: {np.round(expected, 4)}\n"
        "Check that you are computing mean and std correctly."
    )

    # Edge case: all equal rewards -> advantages should all be ~0
    equal_rewards = np.array([0.5, 0.5, 0.5, 0.5])
    result_equal  = my_compute_advantages(equal_rewards)
    assert np.allclose(result_equal, 0.0, atol=1e-4), (
        "When all rewards are equal, all advantages should be ~0. "
        "Check that your epsilon (1e-8) prevents division-by-zero."
    )

    print("Exercise 1.1 passed!")

test_exercise_1_1()

---

### Exercise 1.2: Observe the Effect of Clip Epsilon on Training Stability

**Task:** Run the GRPO training loop three times using `epsilon` values of `0.05`, `0.2`, and `0.8`. Plot all three reward curves on one figure. Answer the conceptual question in the comment at the bottom of the exercise cell.

**Expected Output:** Three labeled reward curves. The `epsilon=0.8` curve should show higher variance than `epsilon=0.05`.

**Hints:**

<details>
<summary>Hint 1</summary>
Copy the training loop from Section 4. Change only the epsilon value for each run. Wrap the loop in a `for eps in EPSILON_VALUES:` block.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Store results as `epsilon_results[eps] = np.array(step_mean_rewards)`. Reset logits to `np.zeros(4)` at the start of each `eps` iteration so runs are independent.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Train with three different epsilon values and collect reward histories.

EPSILON_VALUES = [0.05, 0.2, 0.8]
N_STEPS_EPS   = 60
rng_eps       = np.random.default_rng(SEED)  # fresh rng for a fair comparison

epsilon_results = {}  # epsilon_results[eps] = np.array of mean rewards per step

for eps in EPSILON_VALUES:
    # TODO: run the training loop here for N_STEPS_EPS steps
    # Store the per-step mean reward array in epsilon_results[eps]
    pass

# TODO: plot all three reward curves on one matplotlib figure
# Label each curve with its epsilon value

# ANSWER THIS QUESTION (edit this comment):
# What happens to training stability when epsilon is very small (0.05)?
# What happens to update magnitude when epsilon is very large (0.8)?
# YOUR ANSWER:

The test below checks that you collected results for all three epsilon values and that each run actually learned (mean reward above a minimum threshold).

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_1_2():
    assert len(epsilon_results) == 3, (
        f"Expected results for 3 epsilon values, found {len(epsilon_results)}. "
        "Populate epsilon_results for each of [0.05, 0.2, 0.8]."
    )
    for eps in EPSILON_VALUES:
        assert eps in epsilon_results, (
            f"Missing results for epsilon={eps}. "
            "Store reward history as epsilon_results[eps]."
        )
        arr = np.array(epsilon_results[eps])
        assert arr.ndim == 1, (
            f"epsilon_results[{eps}] should be a 1-D array of mean rewards, "
            f"got shape {arr.shape}."
        )
        assert len(arr) == N_STEPS_EPS, (
            f"Expected {N_STEPS_EPS} steps, got {len(arr)} for epsilon={eps}. "
            "Run the loop for exactly N_STEPS_EPS iterations."
        )
        assert arr.mean() > 0.2, (
            f"Mean reward for epsilon={eps} is {arr.mean():.3f}, which is very low. "
            "Check that reward_function is being called for each sampled index."
        )

    # Small epsilon -> smoother (lower variance) reward curve
    var_small = np.array(epsilon_results[0.05]).var()
    var_large = np.array(epsilon_results[0.8]).var()
    assert var_small <= var_large * 3.0, (
        f"Expected small epsilon to produce lower or similar variance to large epsilon. "
        f"Got var(eps=0.05)={var_small:.4f}, var(eps=0.8)={var_large:.4f}. "
        "Check that you reset policy logits to zeros at the start of each run."
    )

    print("Exercise 1.2 passed!")

test_exercise_1_2()

---

### Exercise 1.3: Implement a Complete GRPO Step with KL Penalty

**Task:** Implement `my_grpo_step` — one complete GRPO update that:
1. Samples G completions from the current policy
2. Scores each with `reward_function`
3. Computes advantages using `my_compute_advantages` (your Exercise 1.1 implementation)
4. Applies the update with KL penalty via `grpo_update_with_kl`
5. Returns `(new_logits, mean_reward, kl)`

This is exactly the function you would call inside a real GRPO training loop.

**Expected Output:** A tuple `(np.ndarray, float, float)`. A 50-step training loop using your function must show higher late-training reward than early-training reward.

**Hints:**

<details>
<summary>Hint 1</summary>
Look at the training loop in Section 5 — your function is exactly one iteration of that loop.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Call sequence: `sample_completions` -> list comprehension with `reward_function` -> `my_compute_advantages` -> `grpo_update_with_kl`. Return `new_logits, float(rewards.mean()), kl`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def my_grpo_step(logits, ref_logits, rng, G=4, epsilon=0.2, beta=0.1, learning_rate=0.4):
    """
    Execute one complete GRPO update step with KL penalty.

    Parameters
    ----------
    logits : np.ndarray, shape (n_actions,)
        Current policy logits
    ref_logits : np.ndarray, shape (n_actions,)
        Reference policy logits (frozen)
    rng : np.random.Generator
    G : int
        Number of completions to sample per step
    epsilon : float
        Clip range for surrogate objective
    beta : float
        KL penalty coefficient
    learning_rate : float

    Returns
    -------
    new_logits : np.ndarray
        Updated policy logits
    mean_reward : float
        Mean reward across the G sampled completions
    kl : float
        KL divergence from reference policy after update
    """
    # Step 1: Sample G completions from the current policy
    sampled_indices, old_probs = None, None  # replace

    # Step 2: Score each sampled completion
    rewards = None  # replace — use reward_function and a list comprehension or loop

    # Step 3: Compute advantages using YOUR implementation from Exercise 1.1
    advantages = None  # replace

    # Step 4: Apply GRPO update with KL penalty
    new_logits, kl = None, None  # replace

    mean_reward = None  # replace

    return new_logits, mean_reward, kl

The test below checks the return types, that logits actually change, that reward is in range, and that running 50 steps of your function produces genuine learning.

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_1_3():
    test_rng    = np.random.default_rng(99)
    test_logits = np.zeros(4)
    test_ref    = np.zeros(4)

    result = my_grpo_step(test_logits, test_ref, test_rng)

    assert result is not None, (
        "my_grpo_step returned None. Replace all None placeholders with real code."
    )
    assert len(result) == 3, (
        f"Expected a tuple of length 3 (logits, mean_reward, kl), got length {len(result)}."
    )

    new_logits, mean_reward, kl = result

    assert isinstance(new_logits, np.ndarray), (
        f"new_logits should be np.ndarray, got {type(new_logits)}."
    )
    assert new_logits.shape == (4,), (
        f"new_logits shape should be (4,), got {new_logits.shape}."
    )
    assert not np.allclose(new_logits, test_logits), (
        "Logits did not change after the update. "
        "Check that grpo_update_with_kl is being called and its return value stored."
    )

    assert isinstance(mean_reward, (float, np.floating)), (
        f"mean_reward should be a float, got {type(mean_reward)}."
    )
    assert 0.0 <= mean_reward <= 1.0, (
        f"mean_reward={mean_reward:.4f} is outside [0, 1]. "
        "Check that reward_function is called for each sampled index."
    )

    assert isinstance(kl, (float, np.floating)), (
        f"kl should be a float, got {type(kl)}."
    )
    assert kl >= 0.0, (
        f"KL divergence must be non-negative, got kl={kl:.6f}. "
        "Check your kl_divergence implementation."
    )

    # Run a short training loop and verify reward actually improves
    loop_rng    = np.random.default_rng(7)
    loop_logits = np.zeros(4)
    loop_ref    = np.zeros(4)
    early_rewards = []
    late_rewards  = []

    for step in range(50):
        loop_logits, r, _ = my_grpo_step(loop_logits, loop_ref, loop_rng)
        if step < 10:
            early_rewards.append(r)
        elif step >= 40:
            late_rewards.append(r)

    assert np.mean(late_rewards) > np.mean(early_rewards), (
        f"Reward did not improve: early={np.mean(early_rewards):.3f}, "
        f"late={np.mean(late_rewards):.3f}. "
        "The loop should converge to higher reward over 50 steps. "
        "Make sure you are using my_compute_advantages (not returning None) "
        "and that you pass new_logits back into my_grpo_step each iteration."
    )

    print("Exercise 1.3 passed!")
    print(f"  Early mean reward (steps  0-9):  {np.mean(early_rewards):.3f}")
    print(f"  Late  mean reward (steps 40-49): {np.mean(late_rewards):.3f}")

test_exercise_1_3()

---

## Summary

### Key Takeaways

1. **GRPO eliminates the critic.** By normalizing rewards across a group of sampled completions, GRPO computes advantages without a separate value model. The group is its own baseline.

2. **Advantages are z-scores.** $A_i = (r_i - \mu_r) / \sigma_r$ — positive for above-average completions, negative for below-average. This single formula drives the policy update.

3. **The clipped surrogate prevents catastrophic updates.** Constraining the probability ratio $\rho_i = \pi_\theta(a) / \pi_{\text{old}}(a)$ to $[1-\varepsilon, 1+\varepsilon]$ keeps each step conservative. Typical $\varepsilon = 0.2$.

4. **KL divergence penalty prevents mode collapse.** Without it, the policy rapidly collapses to a single action even when that action's superiority is marginal. The coefficient $\beta$ controls how tightly the policy is anchored to the reference model.

5. **Policy entropy is a training health metric.** Rising reward plus decreasing entropy signals genuine learning. Entropy collapsing to near-zero within the first few steps signals mode collapse — increase $\beta$.

### What Is Next

- **Module 02 — The ART Framework:** How GRPO is applied to train reasoning agents, with process-level rewards assigned at intermediate reasoning steps rather than only at the final answer.
- **Module 03 — RULER Rewards:** Designing reward functions for complex agentic tasks — correctness signals, format compliance, tool-use verification.

### Additional Resources

- Shao et al., "DeepSeekMath" (2024) — the paper that introduced GRPO
- DeepSeek-R1 paper: "DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning" (2025) — GRPO applied at scale to train a reasoning model
- Schulman et al., "Proximal Policy Optimization Algorithms" (2017) — the PPO paper that GRPO's clipped objective is derived from